# Week 3: RAG Pipeline — M&A Due Diligence
Multi-stage retrieval pipeline: `RecursiveCharacterTextSplitter → all-MiniLM-L6-v2 → ChromaDB → CrossEncoder → Gemini`

In [9]:
import sys, os
sys.path.append(os.path.abspath('..'))

from src.ingestion.pdf_loader import load_pdf
from src.ingestion.chunker import chunk_document
from src.ingestion.embedder import embed_chunks, embed_query
from src.tools.vector_store import retrieve, collection_stats
from src.tools.reranker import rerank
from src.prompts.rag_prompts import rag_query_prompt
from src.llm import gemini_client

print('Imports successful')

Imports successful


## Step 1: Load, Chunk, and Embed PDFs

In [11]:
# Inspect ChromaDB stats
stats = collection_stats(persist_dir='../outputs/chromadb')
print(stats)

{'collection_name': 'ma_due_diligence', 'total_chunks': 2595}


## Step 2: Retrieve + Rerank + Generate

### Query: What are the most significant litigation risks and legal contingencies for each company?

In [12]:
query = "What are the most significant litigation risks and legal contingencies for each company?"
q_emb = embed_query(query)
companies = ["Apple", "Microsoft", "Nvidia"]
all_top_chunks = []
for company in companies:
    candidates = retrieve(q_emb, top_k=10, persist_dir='../outputs/chromadb', where={"company": company})
    top_comp = rerank(query, candidates, top_k=2)
    all_top_chunks.extend(top_comp)

all_top_chunks.sort(key=lambda x: x['rerank_score'], reverse=True)
print(f'Final Chunks after per-company reranking:')
for i, c in enumerate(all_top_chunks, 1):
    print(f"  [{i}] {c['metadata']['company']} | {c['metadata']['section']} | score={c['rerank_score']:.4f}")

Final Chunks after per-company reranking:
  [1] Apple | Business_Overview | score=0.2907
  [2] Microsoft | Business_Overview | score=-0.1288
  [3] Apple | Legal_Proceedings | score=-1.0511
  [4] Nvidia | Risk_Factors | score=-3.2716
  [5] Microsoft | Business_Overview | score=-3.3504
  [6] Nvidia | General | score=-6.2993


In [13]:
prompt = rag_query_prompt(query, all_top_chunks)
# Uncomment to call Gemini
response = gemini_client.query(prompt)
print(response)

Based on the provided sources, the most significant litigation risks and legal contingencies for each company are:

**Apple:**
*   Apple is currently subject to antitrust investigations and litigation in various jurisdictions globally, including civil antitrust lawsuits in the U.S. alleging monopolization or attempted monopolization. These proceedings and claims could have a material adverse impact on the Company’s business, results of operations, financial condition, and stock price [Source 3].
*   The company also faces general legal and regulatory challenges associated with operating in new businesses, regions, or countries, and potential greater-than-expected liabilities and expenses from transactions like investments and acquisitions [Source 1].

**Microsoft:**
*   Microsoft is subject to a variety of new, existing, and evolving legal and regulatory requirements globally, which could adversely affect its results of operations [Source 2].
*   The company may experience new and nove

### Query: Compare the unrecognized tax benefit positions across Apple, Microsoft, and Nvidia.

In [14]:
query = "Compare the unrecognized tax benefit positions across Apple, Microsoft, and Nvidia."
q_emb = embed_query(query)
companies = ["Apple", "Microsoft", "Nvidia"]
all_top_chunks = []
for company in companies:
    candidates = retrieve(q_emb, top_k=10, persist_dir='../outputs/chromadb', where={"company": company})
    top_comp = rerank(query, candidates, top_k=2)
    all_top_chunks.extend(top_comp)

all_top_chunks.sort(key=lambda x: x['rerank_score'], reverse=True)
print(f'Final Chunks after per-company reranking:')
for i, c in enumerate(all_top_chunks, 1):
    print(f"  [{i}] {c['metadata']['company']} | {c['metadata']['section']} | score={c['rerank_score']:.4f}")

Final Chunks after per-company reranking:
  [1] Apple | General | score=3.6471
  [2] Apple | Financial_Statements | score=0.0001
  [3] Nvidia | Income_Tax | score=-0.2935
  [4] Nvidia | Financial_Statements | score=-1.1438
  [5] Microsoft | Income_Tax | score=-7.7749
  [6] Microsoft | Business_Overview | score=-8.9175


In [15]:
prompt = rag_query_prompt(query, all_top_chunks)
# Uncomment to call Gemini
response = gemini_client.query(prompt)
print(response)

Based on the provided context:

**Apple:**
As of September 27, 2025, Apple's total gross unrecognized tax benefits amounted to $23.2 billion. Of this, $10.6 billion, if recognized, would impact the company’s effective tax rate. In the prior year, as of September 28, 2024, the total gross unrecognized tax benefits were $22.0 billion, with $10.8 billion impacting the effective tax rate if recognized [Source 1, Source 2].

**Microsoft:**
The provided context does not contain sufficient information regarding Microsoft's unrecognized tax benefit positions.

**Nvidia:**
The provided context outlines Nvidia's policy for recognizing tax benefits, stating that a benefit from a tax position is recognized only if it is more-likely-than-not that the position would be sustained upon audit based solely on its technical merits. Nvidia's policy also includes interest and penalties related to unrecognized tax benefits as a component of income tax expense [Source 3]. While Source 4 mentions "A reconcili

### Query: What are the largest contractual purchase obligations and how do they compare?

In [16]:
query = "What are the largest contractual purchase obligations and how do they compare?"
q_emb = embed_query(query)
companies = ["Apple", "Microsoft", "Nvidia"]
all_top_chunks = []
for company in companies:
    candidates = retrieve(q_emb, top_k=10, persist_dir='../outputs/chromadb', where={"company": company})
    top_comp = rerank(query, candidates, top_k=2)
    all_top_chunks.extend(top_comp)

all_top_chunks.sort(key=lambda x: x['rerank_score'], reverse=True)
print(f'Final Chunks after per-company reranking:')
for i, c in enumerate(all_top_chunks, 1):
    print(f"  [{i}] {c['metadata']['company']} | {c['metadata']['section']} | score={c['rerank_score']:.4f}")

Final Chunks after per-company reranking:
  [1] Apple | General | score=1.8207
  [2] Apple | General | score=1.4274
  [3] Nvidia | General | score=-2.4462
  [4] Nvidia | General | score=-4.1707
  [5] Microsoft | General | score=-6.5544
  [6] Microsoft | General | score=-8.1831


In [17]:
prompt = rag_query_prompt(query, all_top_chunks)
# Uncomment to call Gemini
response = gemini_client.query(prompt)
print(response)

The largest contractual purchase obligations identified in the provided sources are for Apple.

Apple has "Other Purchase Obligations" totaling $14.8 billion as of September 27, 2025 [Source 1]. These primarily consist of noncancelable obligations to acquire capital assets (including for product manufacturing) and noncancelable obligations related to supplier arrangements, licensed intellectual property and content, and distribution rights [Source 1]. Of this amount, $7.0 billion is payable within 12 months [Source 1].

Apple also has "unconditional purchase obligations" totaling $13.308 billion as of September 27, 2025 [Source 2]. These primarily consist of supplier arrangements, licensed intellectual property and content, and distribution rights [Source 2].

Comparing these, the "Other Purchase Obligations" of $14.8 billion are larger than the "unconditional purchase obligations" of $13.308 billion for Apple.

The provided context does not contain sufficient information to identify o

## Saved Results
All results (JSON + markdown report) saved to `outputs/responses/` and `outputs/reports/`.